Tutorial on buildin simple chatbot with LangChain. Based on:

https://python.langchain.com/docs/use_cases/chatbots/

In [1]:
import os 
from dotenv import load_dotenv

# Define API key for OPenAI
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")


In [ ]:
## Load default OpenAI chatbot
from langchain_openai import ChatOpenAI
chat = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)


In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

## Define prompt
prompt = "Translate this sentence from English to French: I love programming."

# Invoke the chatbot using a hummanMessage to reecive an AIMessage
response = chat.invoke(
    [
        HumanMessage(
            content=prompt
        )
    ]
)

## Take the content from the output
print(response.content)

J'aime programmer.


## We can assign a systemMessage, which will define the role of the chatbot.


In [4]:

response = chat.invoke(
    [
        SystemMessage(
            content = 'You are a helpfull assistant that translates English to french'
        ),
        HumanMessage(
            content="I love programming and everything else"
        )
    ]
)

print(response.content)

J'aime la programmation et tout le reste.


### Chatbots do not interact with the real world...

Chatbots cannot interact with anything outside of their enviroment, they will only generate text from the knowledge they were trained on. Additionally, they do not have acces to external codes or applicatoin that could allow interaction with outside world.

For example, if we ask the chatbot about files in the current directory it will no be able to determine them since it cannot interact with the real world.

In [5]:
chatbot_test = chat.invoke(
    [HumanMessage(content="What are the files in the current directory?")
     ])

print(chatbot_test.content)

I don't have the ability to access or view files in your current directory or any external system. However, you can check the files in your current directory by using commands in your terminal or command prompt. 

For example:

- On Windows, you can use:
  ```
  dir
  ```

- On macOS or Linux, you can use:
  ```
  ls
  ```

These commands will list the files and directories in your current working directory.


# An agent is just a chatbot that can use tools to complete tasks.

Tools being code that allows the LLM to interact with systems outside of its session. You can use pre-defined tools or define custom ones. There are also different agent types which have different purposes.

https://python.langchain.com/docs/integrations/tools/ <br>
https://python.langchain.com/docs/modules/agents/tools/custom_tools <br>
https://python.langchain.com/docs/modules/agents/agent_types/<br>
https://react-lm.github.io/

Lets start with a simple agent that can use a tool for Shell command sand bash scripting.The LLM can use it to execute any shell commands. A common use case for this is letting the LLM interact with your local file system.

Note: Shell tool does not work properly with Windows OS.

In [6]:
# Import tool pre-defined tool from langchain
from langchain.tools import ShellTool
shell_tool = ShellTool()

# Print tool description 
print(shell_tool.name)
print(shell_tool.description)
(shell_tool.args)

terminal
Run shell commands on this Linux machine.


{'commands': {'anyOf': [{'type': 'string'},
   {'items': {'type': 'string'}, 'type': 'array'}],
  'description': 'List of shell commands to run. Deserialized using json.loads',
  'title': 'Commands'}}

In [7]:
## We can use the tool directly without a LLM

## Example usage of shell tool
print(shell_tool.run({"commands": ["echo 'Hello World!'", "pwd"]}))


Executing command:
 ["echo 'Hello World!'", 'pwd']
Hello World!
/scratch365/omendibl/Molec_Mindset/DynaMate_V3/tutorials



/afs/crc.nd.edu/user/o/omendibl/.conda/envs/dynamate/lib/python3.10/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Defining and Agent

This involves defining the LLM model, the list of tools the agent has access to, and the promp template. Then yo need to define the type of Agent (Reason Action in this case) and the agent executor.

In [8]:
## Load default OpenAI chatbot
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor,create_react_agent # To load simple ReAct agent. Reason an act
from langchain import hub

## Define LLM
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Define list of tools the LLM is going to use 
tools = [shell_tool]

# Get the template prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react")

## Read the prompt template 
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


/afs/crc.nd.edu/user/o/omendibl/.conda/envs/dynamate/lib/python3.10/site-packages/langsmith/client.py:278: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [9]:
## Construct the ReAct agent by defining the llm, tools and prompt template
shell_Agent = create_react_agent(llm=llm,tools=tools,prompt=prompt)

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=shell_Agent, tools=tools, verbose=True)

In [10]:
## Run the agent executor

input = 'What are the files in the current directory?'
response = agent_executor.invoke({"input":str(input)})
response



> Entering new AgentExecutor chain...
I need to list the files in the current directory to answer the question. 
Action: terminal 
Action Input: ls Executing command:
 ls
0_chatbot_vs_agent.ipynb


/afs/crc.nd.edu/user/o/omendibl/.conda/envs/dynamate/lib/python3.10/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


I now know the final answer
Final Answer: The files in the current directory are: 0_chatbot_vs_agent.ipynb

> Finished chain.


{'input': 'What are the files in the current directory?',
 'output': 'The files in the current directory are: 0_chatbot_vs_agent.ipynb'}